In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR = PROJECT_ROOT / "data" / "raw"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

%reload_ext autoreload
%autoreload 2

print("Project root:", PROJECT_ROOT)
print("Source dir:", SRC_DIR)
print("Processed dir:", PROCESSED_DIR)

Project root: /workspaces/GraphRepresentationLearning
Source dir: /workspaces/GraphRepresentationLearning/src
Processed dir: /workspaces/GraphRepresentationLearning/data/processed


In [2]:
from load_data import load_hpo

from analyze_embeddings import(
    load_disease_embeddings,
    compute_nearest_neighbors,
    explain_neighbor_pairs_with_shared_hpo,
    summarize_neighbor_explainability,
    compute_umap_projection,
    save_dataframe,
)

/workspaces/GraphRepresentationLearning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
TRANSE_EMBEDDINGS_PATH = PROCESSED_DIR / "disease_embeddings_transe.csv"
HPOA_FILTERED_PATH = PROCESSED_DIR / "hpoa_filtered.csv"
HPO_PATH = RAW_DIR / "hp.obo"#

transe_embeddings = load_disease_embeddings(
    TRANSE_EMBEDDINGS_PATH,
    id_column="entity_id",
)

hpoa_filtered = pd.read_csv(HPOA_FILTERED_PATH, dtype=str)
hpo = load_hpo(HPO_PATH)

print("TransE embeddings:", transe_embeddings.shape)
print("Filtered HPOA:", hpoa_filtered.shape)
print("HPO nodes:", hpo.number_of_nodes())

transe_embeddings.head()

TransE embeddings: (1000, 35)
Filtered HPOA: (20178, 12)
HPO nodes: 19389


,entity_id,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,...,dim_24,dim_25,dim_26,dim_27,dim_28,dim_29,dim_30,dim_31,entity_type,label
0,DECIPHER:21,0.042787,0.248684,-0.203058,-0.046804,-0.283181,-0.067965,-0.023819,-0.003923,0.136725,...,-0.326444,-0.131847,0.220800,-0.042644,0.154087,-0.094088,0.079420,0.322761,disease,Miller-Dieker syndrome (MDS)
1,DECIPHER:3,0.087997,-0.206632,-0.120176,-0.352266,-0.101608,-0.221718,-0.040925,0.071674,0.411519,...,-0.250823,0.044286,-0.218914,-0.237703,0.038430,-0.260760,-0.184132,0.061635,disease,Williams-Beuren Syndrome (WBS)
2,DECIPHER:45,-0.086892,0.143391,-0.149582,-0.118088,-0.122956,-0.098960,0.071973,-0.390183,0.161120,...,-0.196338,0.023438,0.173035,-0.129244,0.025186,0.032387,-0.069286,-0.120627,disease,Xq28 (MECP2) duplication
3,DECIPHER:54,-0.145245,0.049770,-0.195604,-0.392991,-0.124197,0.341346,-0.086276,0.176357,0.059678,...,0.132165,0.159629,0.146437,0.251457,0.079966,-0.143056,-0.310975,-0.025079,disease,Angelman syndrome (Type 2)
4,DECIPHER:81,-0.244729,0.034498,-0.122347,-0.170408,-0.101192,-0.014939,-0.052999,-0.087859,0.007928,...,-0.321677,0.197775,0.062999,-0.087581,0.103390,-0.009194,-0.207620,0.375099,disease,15q26 overgrowth syndrome


In [4]:
transe_neighbors = compute_nearest_neighbors(
    embeddings=transe_embeddings,
    top_k=10,
    id_column="entity_id",
    similarity_column="cosine_similarity",
)

transe_neighbors.head(10)

,disease_id,disease_label,neighbor_id,neighbor_label,rank,cosine_similarity
0,DECIPHER:21,Miller-Dieker syndrome (MDS),OMIM:620194,"Neurodevelopmental disorder with poor growth, ...",1,0.621034
1,DECIPHER:21,Miller-Dieker syndrome (MDS),OMIM:609975,"Hyperinsulinemic hypoglycemia, familial, 4",2,0.591657
2,DECIPHER:21,Miller-Dieker syndrome (MDS),OMIM:612530,Chromosome 1q41-q42 deletion syndrome,3,0.568496
3,DECIPHER:21,Miller-Dieker syndrome (MDS),OMIM:619980,Braddock-Carey syndrome 1,4,0.562556
4,DECIPHER:21,Miller-Dieker syndrome (MDS),ORPHA:411536,Mild phosphoribosylpyrophosphate synthetase su...,5,0.558425
5,DECIPHER:21,Miller-Dieker syndrome (MDS),ORPHA:166100,Autosomal dominant otospondylomegaepiphyseal d...,6,0.552337
6,DECIPHER:21,Miller-Dieker syndrome (MDS),ORPHA:300493,Sagliker syndrome,7,0.528506
7,DECIPHER:21,Miller-Dieker syndrome (MDS),ORPHA:215,Congenital stationary night blindness,8,0.511869
8,DECIPHER:21,Miller-Dieker syndrome (MDS),OMIM:159400,"Myasthenia, limb-girdle, autoimmune",9,0.510574
9,DECIPHER:21,Miller-Dieker syndrome (MDS),OMIM:113500,Brachyolmia type 3,10,0.506288


In [5]:
transe_neighbor_explanations = explain_neighbor_pairs_with_shared_hpo(
    neighbors=transe_neighbors,
    hpoa=hpoa_filtered,
    hpo=hpo,
    max_terms_per_pair=20,
)

transe_neighbor_explanations.head(20)

,disease_id,neighbor_id,rank,cosine_similarity,shared_hpo_id,shared_hpo_label,num_shared_hpo_terms,disease_hpo_count,neighbor_hpo_count
0,DECIPHER:21,OMIM:620194,1,0.621034,HP:0000252,Microcephaly,1,5,28
1,DECIPHER:21,OMIM:609975,2,0.591657,None,None,0,5,6
2,DECIPHER:21,OMIM:612530,3,0.568496,HP:0000252,Microcephaly,2,5,47
3,DECIPHER:21,OMIM:612530,3,0.568496,HP:0002007,Frontal bossing,2,5,47
4,DECIPHER:21,OMIM:619980,4,0.562556,HP:0000252,Microcephaly,1,5,32
5,DECIPHER:21,ORPHA:411536,5,0.558425,None,None,0,5,10
6,DECIPHER:21,ORPHA:166100,6,0.552337,None,None,0,5,12
7,DECIPHER:21,ORPHA:300493,7,0.528506,HP:0002007,Frontal bossing,1,5,13
8,DECIPHER:21,ORPHA:215,8,0.511869,None,None,0,5,14
9,DECIPHER:21,OMIM:159400,9,0.510574,None,None,0,5,11


In [6]:
transe_explainablity_summary = summarize_neighbor_explainability(
    neighbors=transe_neighbors,
    hpoa=hpoa_filtered,
)

transe_explainablity_summary.head(20)

,disease_id,disease_label,neighbor_id,neighbor_label,rank,cosine_similarity,num_shared_hpo_terms,jaccard_hpo_similarity,disease_hpo_count,neighbor_hpo_count
0,DECIPHER:21,Miller-Dieker syndrome (MDS),OMIM:620194,"Neurodevelopmental disorder with poor growth, ...",1,0.621034,1,0.031250,5,28
1,DECIPHER:21,Miller-Dieker syndrome (MDS),OMIM:609975,"Hyperinsulinemic hypoglycemia, familial, 4",2,0.591657,0,0.000000,5,6
2,DECIPHER:21,Miller-Dieker syndrome (MDS),OMIM:612530,Chromosome 1q41-q42 deletion syndrome,3,0.568496,2,0.040000,5,47
3,DECIPHER:21,Miller-Dieker syndrome (MDS),OMIM:619980,Braddock-Carey syndrome 1,4,0.562556,1,0.027778,5,32
4,DECIPHER:21,Miller-Dieker syndrome (MDS),ORPHA:411536,Mild phosphoribosylpyrophosphate synthetase su...,5,0.558425,0,0.000000,5,10
5,DECIPHER:21,Miller-Dieker syndrome (MDS),ORPHA:166100,Autosomal dominant otospondylomegaepiphyseal d...,6,0.552337,0,0.000000,5,12
6,DECIPHER:21,Miller-Dieker syndrome (MDS),ORPHA:300493,Sagliker syndrome,7,0.528506,1,0.058824,5,13
7,DECIPHER:21,Miller-Dieker syndrome (MDS),ORPHA:215,Congenital stationary night blindness,8,0.511869,0,0.000000,5,14
8,DECIPHER:21,Miller-Dieker syndrome (MDS),OMIM:159400,"Myasthenia, limb-girdle, autoimmune",9,0.510574,0,0.000000,5,11
9,DECIPHER:21,Miller-Dieker syndrome (MDS),OMIM:113500,Brachyolmia type 3,10,0.506288,0,0.000000,5,13


In [7]:
transe_explainablity_summary.sort_values(
    by="cosine_similarity",
    ascending=False,
).head(20)[
    [
        "disease_label",
        "neighbor_label",
        "rank",
        "cosine_similarity",
        "num_shared_hpo_terms",
        "jaccard_hpo_similarity",
        "disease_hpo_count",
        "neighbor_hpo_count"
    ]
]

,disease_label,neighbor_label,rank,cosine_similarity,num_shared_hpo_terms,jaccard_hpo_similarity,disease_hpo_count,neighbor_hpo_count
2060,"Spinal and bulbar muscular atrophy, X-linked 1",Papillary tumor of the pineal region,1,0.824360,0,0.000000,16,14
7750,Papillary tumor of the pineal region,"Spinal and bulbar muscular atrophy, X-linked 1",1,0.824360,0,0.000000,14,16
3730,"Leukodystrophy, hypomyelinating, 8, with or wi...","Intellectual developmental disorder, autosomal...",1,0.807729,10,0.131579,37,49
3120,"Intellectual developmental disorder, autosomal...","Leukodystrophy, hypomyelinating, 8, with or wi...",1,0.807729,10,0.131579,49,37
2000,"Chronic granulomatous disease, X-linked","Chronic granulomatous disease 2, autosomal rec...",1,0.786723,23,0.522727,31,36
1240,"Chronic granulomatous disease 2, autosomal rec...","Chronic granulomatous disease, X-linked",1,0.786723,23,0.522727,36,31
1970,"Spinocerebellar ataxia, X-linked 1",Dystonia 16,1,0.778714,3,0.090909,15,21
3190,Dystonia 16,"Spinocerebellar ataxia, X-linked 1",1,0.778714,3,0.090909,21,15
5820,Gastrointestinal defects and immunodeficiency ...,Diaphanospondylodysostosis,1,0.778441,4,0.048780,40,46
2730,Diaphanospondylodysostosis,Gastrointestinal defects and immunodeficiency ...,1,0.778441,4,0.048780,46,40


In [8]:
transe_explainablity_summary[
    transe_explainablity_summary["num_shared_hpo_terms"] == 0
].sort_values(
    by="cosine_similarity",
    ascending=False,
).head(20)[
    [
        "disease_label",
        "neighbor_label",
        "rank",
        "cosine_similarity",
        "num_shared_hpo_terms",
        "jaccard_hpo_similarity"
    ]
]

,disease_label,neighbor_label,rank,cosine_similarity,num_shared_hpo_terms,jaccard_hpo_similarity
7750,Papillary tumor of the pineal region,"Spinal and bulbar muscular atrophy, X-linked 1",1,0.824360,0,0.0
2060,"Spinal and bulbar muscular atrophy, X-linked 1",Papillary tumor of the pineal region,1,0.824360,0,0.0
6550,Neuroendocrine tumor of anal canal,Familial intestinal malrotation,1,0.776608,0,0.0
9010,Familial intestinal malrotation,Neuroendocrine tumor of anal canal,1,0.776608,0,0.0
9380,Congenital erythropoietic porphyria,Autosomal recessive faciodigitogenital syndrome,1,0.770262,0,0.0
7240,Autosomal recessive faciodigitogenital syndrome,Congenital erythropoietic porphyria,1,0.770262,0,0.0
2640,"Symphalangism, distal, with microdontia, denta...",Incontinentia pigmenti,1,0.761018,0,0.0
2030,Incontinentia pigmenti,"Symphalangism, distal, with microdontia, denta...",1,0.761018,0,0.0
2360,"Migraine, familial hemiplegic, 2",Lethal Kniest-like dysplasia,1,0.757803,0,0.0
7650,Lethal Kniest-like dysplasia,"Migraine, familial hemiplegic, 2",1,0.757803,0,0.0


In [9]:
transe_projection = compute_umap_projection(
    embeddings=transe_embeddings,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=5,
    id_column="entity_id",
)

transe_projection.head()

/workspaces/GraphRepresentationLearning/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,node_id,x,y,entity_type,label
0,DECIPHER:21,1.767297,4.519266,disease,Miller-Dieker syndrome (MDS)
1,DECIPHER:3,1.096477,4.854497,disease,Williams-Beuren Syndrome (WBS)
2,DECIPHER:45,3.324550,4.682528,disease,Xq28 (MECP2) duplication
3,DECIPHER:54,2.870575,5.384541,disease,Angelman syndrome (Type 2)
4,DECIPHER:81,1.290076,6.107343,disease,15q26 overgrowth syndrome


In [10]:
fig = px.scatter(
    transe_projection,
    x="x",
    y="y",
    hover_name="label",
    hover_data=["node_id"],
    title="TransE disease embeddings projected with UMAP",
)

fig.show()

In [11]:
save_dataframe(
    transe_neighbors,
    PROCESSED_DIR / "transe_neighbors.csv",
)

save_dataframe(
    transe_neighbor_explanations,
    PROCESSED_DIR / "transe_neighbor_shared_hpo_terms.csv",
)

save_dataframe(
    transe_explainablity_summary,
    PROCESSED_DIR / "transe_neighbor_explainability_summary.csv",
)

save_dataframe(
    transe_projection,
    PROCESSED_DIR / "transe_umap_projection.csv",
)

print("Saved TransE analysis output")

Saved TransE analysis output
